In [1]:
import os
print(os.listdir('/kaggle/input'))

['models', 'competitions']


In [2]:
# ============================================================
# INFERENCE NOTEBOOK — Config
# Edit INFER_THRESH, RADIUS_XY, RADIUS_Z here and re-run Cell 1
# then jump to Cell 5 (no need to reload model) to re-generate submission
# ============================================================
import os, json, glob, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max

DEVICE = 'cpu'
if torch.cuda.is_available():
    try:
        _p = nn.Conv2d(1, 1, 3).cuda()
        _ = _p(torch.zeros(1, 1, 8, 8, device='cuda')).cpu()
        del _p
        DEVICE = 'cuda'
    except Exception as e:
        print(f'GPU unusable: {str(e)[:80]} -> CPU')
print(f'Device: {DEVICE}')

COMP_ROOT = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR   = Path(COMP_ROOT) / 'test'
OUT_DIR    = Path('/kaggle/working')
MODELS_DIR = Path('/kaggle/input/models')   # update to your attached dataset path

# load metadata
meta_path = next(MODELS_DIR.rglob('*.json'), None)
if meta_path is not None:
    with open(meta_path) as f:
        saved_meta = json.load(f)
    print(f'Meta loaded from {meta_path}:')
    print(json.dumps(saved_meta, indent=2))
    BASE       = int(saved_meta.get('base',       16))
    POOL       = int(saved_meta.get('pool',         2))
    N_CONTEXT  = int(saved_meta.get('n_context',   3))
    VAL_RECALL = saved_meta.get('best_val_recall', saved_meta.get('val_recall', 'unknown'))

    # FIX: try 'best_thresh' first (written by training Cell 8 above)
    # fall back to 'detect_thresh' for older checkpoints
    INFER_THRESH = float(
        saved_meta.get('best_thresh',
        saved_meta.get('detect_thresh', 0.35))
    )
    # FIX: load NMS radii from metadata (written by training Cell 8 above)
    NMS_RADIUS_XY = float(saved_meta.get('nms_radius_xy', 6.0 if POOL == 2 else 3.0))
    NMS_RADIUS_Z  = int(  saved_meta.get('nms_radius_z',  3))
else:
    print('No meta JSON -- using defaults')
    BASE          = 16
    POOL          = 2
    N_CONTEXT     = 3
    INFER_THRESH  = 0.35
    NMS_RADIUS_XY = 6.0
    NMS_RADIUS_Z  = 3
    VAL_RECALL    = 'unknown'

# compute dependent parameters
NUCLEUS_DIAM_UM   = 8.0
VOXEL_XY_POOLED   = 0.40625 * POOL
MIN_PEAK_DISTANCE = max(4, int(NUCLEUS_DIAM_UM / VOXEL_XY_POOLED))
MIN_CONSECUTIVE_Z = 3    # FIX: was 2 -- must match training evaluation
MAX_LINK_DIST_UM  = 15.0 # FIX: was 7.0 -- 7um is metric match radius, not linking radius
DIV_MAX_DIST_UM   = 8.0

# ---- load checkpoint ----
CKPT_PATH = next(MODELS_DIR.rglob('unet2d_best.pt'),
            next(MODELS_DIR.rglob('*.pt'), None))
if CKPT_PATH is None:
    raise FileNotFoundError(f'No .pt file found under {MODELS_DIR}')
print(f'Loading: {CKPT_PATH}')

# raw state dict -- load directly into model after model is defined in Cell 2
raw_state_dict = torch.load(CKPT_PATH, map_location='cpu')
print(f'State dict keys: {len(raw_state_dict)} tensors')

SCALE = np.array([1.625, 0.40625, 0.40625])
print(f'SCALE={SCALE} | TEST_DIR exists={TEST_DIR.exists()}')

print(f'\nBASE={BASE} | POOL={POOL} | N_CONTEXT={N_CONTEXT}')
print(f'INFER_THRESH={INFER_THRESH} | val_recall={VAL_RECALL}')
print(f'MIN_PEAK_DISTANCE={MIN_PEAK_DISTANCE}px')
print(f'NMS_RADIUS_XY={NMS_RADIUS_XY}px | NMS_RADIUS_Z={NMS_RADIUS_Z}')
print(f'MIN_CONSECUTIVE_Z={MIN_CONSECUTIVE_Z} | MAX_LINK_DIST_UM={MAX_LINK_DIST_UM}')

Device: cpu
Meta loaded from /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default/4/unet2d_meta.json:
{
  "base": 32,
  "pool": 4,
  "gauss_sigma": 2.0,
  "best_thresh": 0.05,
  "best_val_recall": 0.6621621621621622,
  "min_peak_distance": 5,
  "nucleus_diam_um": 8.0,
  "clahe_clip": 3.0,
  "clahe_grid": 8,
  "w_pos": 15.0,
  "w_bg": 1.0,
  "w_ign": 0.01,
  "z_margin": 5,
  "neg_slices_per_t": 4,
  "epochs_trained": 60
}
Loading: /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default/4/unet2d_best.pt
State dict keys: 76 tensors
SCALE=[1.625   0.40625 0.40625] | TEST_DIR exists=True

BASE=32 | POOL=4 | N_CONTEXT=3
INFER_THRESH=0.05 | val_recall=0.6621621621621622
MIN_PEAK_DISTANCE=4px
NMS_RADIUS_XY=3.0px | NMS_RADIUS_Z=3
MIN_CONSECUTIVE_Z=3 | MAX_LINK_DIST_UM=15.0


In [3]:
# Must match the architecture used in training exactly
# If you changed BASE in the training notebook, update it here too
def _block2d(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
    )

class UNet2D(nn.Module):
    def __init__(self, base=BASE):
        super().__init__()
        b = base
        self.e1   = _block2d(1,   b)    # 1 channel -- 2D model
        self.e2   = _block2d(b,   b*2)
        self.pool = nn.MaxPool2d(2)
        self.bn   = _block2d(b*2, b*4)
        self.u2   = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.d2   = _block2d(b*4, b*2)
        self.u1   = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.d1   = _block2d(b*2, b)
        self.out  = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        b  = self.bn(self.pool(e2))
        d2 = self.d2(torch.cat([self.u2(b), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet2D(base=BASE).to(DEVICE)
model.load_state_dict(raw_state_dict)
model.eval()
print(f'2D UNet loaded. BASE={BASE}, POOL={POOL}, 1-channel input.')

2D UNet loaded. BASE=32, POOL=4, 1-channel input.


In [4]:
_ZC = {}

def read_meta(zp):
    with open(Path(zp) / '0' / 'zarr.json') as f:
        m = json.load(f)
    return dict(shape=tuple(m['shape']), dtype=np.dtype(m['data_type']))

def load_vol(zp, t, meta=None):
    try:
        import zarr
        k = str(zp)
        if k not in _ZC:
            _ZC[k] = zarr.open(k, mode='r')['0']
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None:
            meta = read_meta(zp)
        buf = blosc2.decompress(
            open(Path(zp) / '0' / 'c' / str(t) / '0' / '0' / '0', 'rb').read()
        )
        return np.frombuffer(buf, dtype=meta['dtype']).reshape(meta['shape'][1:])

def pool_xy(vol, f=POOL):
    Z, Y, X = vol.shape
    Y2, X2 = (Y // f) * f, (X // f) * f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2 // f, f, X2 // f, f).mean(axis=(2, 4))

def normalize_slice_clahe(slc, clip=2.0, grid=8):
    lo = float(np.percentile(slc, 2.0))
    hi = float(np.percentile(slc, 99.0))
    if hi <= lo:
        return np.zeros_like(slc, dtype=np.float32)
    scaled = np.clip((slc-lo)/(hi-lo)*255, 0, 255).astype(np.uint8)
    return cv2.createCLAHE(clipLimit=clip,
                            tileGridSize=(grid,grid)).apply(scaled).astype(np.float32)/255.0

def detect_volume_with_scores(pvol, model,
                               thresh=INFER_THRESH,
                               min_dist=MIN_PEAK_DISTANCE,
                               min_consec=MIN_CONSECUTIVE_Z):
    """Single-channel 2D detection matching the 2D model's training format."""
    model.eval()
    Z = pvol.shape[0]
    slice_dets   = {}
    slice_scores = {}

    with torch.no_grad():
        for z in range(Z):
            slc = normalize_slice_clahe(pvol[z])
            xt  = torch.from_numpy(slc[None, None]).to(DEVICE)  # (1,1,H,W) for 2D model
            hm  = torch.sigmoid(model(xt))[0,0].detach().cpu().numpy()
            pk  = peak_local_max(hm, min_distance=min_dist,
                                  threshold_abs=thresh, exclude_border=False)
            slice_dets[z]   = pk
            slice_scores[z] = np.array([hm[p[0],p[1]] for p in pk]) \
                               if len(pk) > 0 else np.array([])

    confirmed = []
    conf_scrs = []
    for z in range(Z):
        pk_z  = slice_dets.get(z,  np.zeros((0,2)))
        scr_z = slice_scores.get(z, np.array([]))
        if len(pk_z) == 0:
            continue
        for idx in range(len(pk_z)):
            det_yx = pk_z[idx]
            consec = 1
            cur_yx = det_yx.copy()
            for dz in range(1, min_consec):
                pk_next = slice_dets.get(z+dz, np.zeros((0,2)))
                if len(pk_next) == 0:
                    break
                dists = np.linalg.norm(pk_next - cur_yx, axis=1)
                ci    = np.argmin(dists)
                if dists[ci] <= 3.0:   # tight Z-consistency for 2D model
                    consec += 1
                    cur_yx  = pk_next[ci]
                else:
                    break
            if consec >= min_consec:
                confirmed.append((z, float(det_yx[0]), float(det_yx[1])))
                conf_scrs.append(float(scr_z[idx]))

    return confirmed, conf_scrs

def nms_3d_weighted(confirmed_dets, scores,
                     radius_xy=3.0,   # 2D model: POOL=4, so 3px = 12 original voxels
                     radius_z=3):
    """NMS with score-weighted centroid. Returns (centroids, aligned_scores)."""
    if len(confirmed_dets) == 0:
        return [], []
    dets  = np.array(confirmed_dets, dtype=np.float64)
    scrs  = np.array(scores,         dtype=np.float64)
    order = np.argsort(scrs)[::-1]
    keep_centroids = []
    keep_scores    = []
    suppressed     = np.zeros(len(dets), dtype=bool)
    for i in order:
        if suppressed[i]:
            continue
        zi, yi, xi = dets[i]
        cluster_idx = []
        for j in range(len(dets)):
            if suppressed[j]:
                continue
            zj, yj, xj = dets[j]
            if abs(zi-zj) <= radius_z and \
               np.sqrt((yi-yj)**2 + (xi-xj)**2) <= radius_xy:
                cluster_idx.append(j)
                suppressed[j] = True
        c_dets = dets[cluster_idx]
        c_scrs = scrs[cluster_idx]
        tw     = c_scrs.sum()
        if tw > 0:
            z_wt = float(np.dot(c_scrs, c_dets[:,0]) / tw)
            y_wt = float(np.dot(c_scrs, c_dets[:,1]) / tw)
            x_wt = float(np.dot(c_scrs, c_dets[:,2]) / tw)
        else:
            z_wt, y_wt, x_wt = float(zi), float(yi), float(xi)
        keep_centroids.append((z_wt, y_wt, x_wt))
        keep_scores.append(float(c_scrs.max()))
    return keep_centroids, keep_scores

print('Cell 4 complete. 2D detection + NMS defined.')
print(f'POOL={POOL} | NMS radius_xy=3.0px = {3*POOL*0.40625:.2f}um physical')

Cell 4 complete. 2D detection + NMS defined.
POOL=4 | NMS radius_xy=3.0px = 4.88um physical


In [5]:
# ---- inference config ----
NUCLEUS_DIAM_UM   = 8.0
VOXEL_XY_POOLED   = 0.40625 * POOL
MIN_PEAK_DISTANCE = max(5, int(NUCLEUS_DIAM_UM / VOXEL_XY_POOLED))
MIN_CONSECUTIVE_Z = 3
NMS_RADIUS_XY     = 6.0
NMS_RADIUS_Z      = 3
INFER_THRESH      = 0.25   # kept at 0.25 -- adaptive per-volume adjustment below

# Z-consistency matching budget: how far a detection can drift in XY
# between consecutive Z-slices and still count as the same nucleus.
# Use 3.0 pooled pixels (~2.4 um) -- a nucleus doesn't move across Z-slices,
# only the imaging plane shifts. Was NMS_RADIUS_XY*2=12 which is way too large.
Z_CONSISTENCY_XY = 3.0

print(f'INFER_THRESH={INFER_THRESH} | MIN_PEAK_DISTANCE={MIN_PEAK_DISTANCE}')
print(f'MIN_CONSECUTIVE_Z={MIN_CONSECUTIVE_Z}')
print(f'NMS_RADIUS_XY={NMS_RADIUS_XY} | NMS_RADIUS_Z={NMS_RADIUS_Z}')
print(f'Z_CONSISTENCY_XY={Z_CONSISTENCY_XY}')

def get_2p5d_stack_clahe(pvol, z, clip=3.0, grid=8):
    """3-channel stack: [z-1, z, z+1] with CLAHE normalization."""
    Z = pvol.shape[0]
    slices = []
    for dz in [-1, 0, 1]:
        z_idx = int(np.clip(z + dz, 0, Z - 1))
        slc   = pvol[z_idx]
        lo    = float(np.percentile(slc, 2.0))
        hi    = float(np.percentile(slc, 99.0))
        if hi <= lo:
            slices.append(np.zeros_like(slc, dtype=np.float32))
            continue
        scaled = np.clip((slc - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)
        clahe  = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
        slices.append(clahe.apply(scaled).astype(np.float32) / 255.0)
    return np.stack(slices, axis=0)   # (3, H, W)

def get_volume_peak(pvol, model, n_sample=8):
    """
    Samples n_sample evenly-spaced Z-slices and returns the median
    peak heatmap value. Used to adapt the detection threshold per volume.
    Single pass only -- does NOT run full detection.
    """
    sample_zs = list(range(0, pvol.shape[0],
                           max(1, pvol.shape[0] // n_sample)))[:n_sample]
    peak_vals = []
    model.eval()
    with torch.no_grad():
        for z in sample_zs:
            stack = get_2p5d_stack_clahe(pvol, z)
            xt    = torch.from_numpy(stack[None]).to(DEVICE)
            hm    = torch.sigmoid(model(xt))[0, 0].detach().cpu().numpy()
            peak_vals.append(float(hm.max()))
    return float(np.median(peak_vals))

def adaptive_threshold(pvol, base_thresh=INFER_THRESH):
    """
    If the model's median peak response on this volume is low,
    the threshold is lowered proportionally so dim datasets still
    get detections (e.g. 44b6_0b24845f which was getting 0 detections
    at thresh=0.35 because its heatmap max never reached that value).

    Logic: threshold = min(base_thresh, vol_max * 0.6)
    This ensures threshold never EXCEEDS the volume's actual peak,
    while also never going below 0.05 (pure noise floor).
    """
    vol_max  = get_volume_peak(pvol, model)
    # set threshold at 60% of the volume's median peak response
    # this guarantees the top ~40% of activations pass the threshold
    adaptive = float(np.clip(vol_max * 0.6, 0.05, base_thresh))
    if adaptive < base_thresh:
        print(f'    [adaptive] vol_peak_median={vol_max:.3f} '
              f'-> thresh={adaptive:.3f} (base={base_thresh:.3f})')
    return adaptive

def detect_volume_with_scores_2p5d(pvol, model,
                                    thresh=None,
                                    min_dist=MIN_PEAK_DISTANCE,
                                    min_consec_z=MIN_CONSECUTIVE_Z,
                                    verbose=False):
    """
    Runs 2.5D UNet slice-by-slice. Returns (confirmed, scores).
    thresh=None triggers adaptive per-volume threshold.
    verbose=False suppresses per-slice debug output during full submission run.
    """
    if thresh is None:
        thresh = adaptive_threshold(pvol)

    model.eval()
    Z = pvol.shape[0]
    slice_dets   = {}
    slice_scores = {}

    with torch.no_grad():
        for z in range(Z):
            stack = get_2p5d_stack_clahe(pvol, z)
            xt    = torch.from_numpy(stack[None]).to(DEVICE)
            hm    = torch.sigmoid(model(xt))[0, 0].detach().cpu().numpy()

            # only print debug output when explicitly requested
            if verbose and (z == 0 or z % 20 == 0):
                print(f'      z={z:3d} hm=[{hm.min():.3f},{hm.max():.3f}] '
                      f'thresh={thresh:.3f}')

            pk = peak_local_max(hm, min_distance=min_dist,
                                 threshold_abs=thresh, exclude_border=False)
            slice_dets[z]   = pk
            # build scores aligned with pk -- same length guaranteed
            slice_scores[z] = np.array([hm[p[0], p[1]] for p in pk]) \
                               if len(pk) > 0 else np.array([])

    confirmed = []
    conf_scrs = []
    for z in range(Z):
        pk_z  = slice_dets.get(z,  np.zeros((0, 2)))
        scr_z = slice_scores.get(z, np.array([]))
        if len(pk_z) == 0:
            continue
        for idx in range(len(pk_z)):
            det_yx = pk_z[idx]
            consec = 1
            cur_yx = det_yx.copy()
            for dz in range(1, min_consec_z):
                pk_next = slice_dets.get(z + dz, np.zeros((0, 2)))
                if len(pk_next) == 0:
                    break
                dists = np.linalg.norm(pk_next - cur_yx, axis=1)
                ci    = np.argmin(dists)
                # FIX: use Z_CONSISTENCY_XY not NMS_RADIUS_XY*2
                # Z-consistency only needs a small drift budget (~1 nucleus radius)
                # NMS_RADIUS_XY*2 = 12px was merging detections from different cells
                if dists[ci] <= Z_CONSISTENCY_XY:
                    consec += 1
                    cur_yx  = pk_next[ci]
                else:
                    break
            if consec >= min_consec_z:
                confirmed.append((z, float(det_yx[0]), float(det_yx[1])))
                # scr_z[idx] is safe: scr_z and pk_z are built from the same pk
                conf_scrs.append(float(scr_z[idx]))

    return confirmed, conf_scrs

def nms_3d_weighted(confirmed_dets, scores,
                     radius_xy=NMS_RADIUS_XY,
                     radius_z=NMS_RADIUS_Z):
    """
    3D NMS with score-weighted centroid.
    Returns (centroids, aligned_scores) -- both lists same length.
    CRITICAL: centroids[i] and aligned_scores[i] always correspond.
    The calling code MUST use aligned_scores[i], not the original raw scores.
    """
    if len(confirmed_dets) == 0:
        return [], []

    dets  = np.array(confirmed_dets, dtype=np.float64)
    scrs  = np.array(scores,         dtype=np.float64)
    order = np.argsort(scrs)[::-1]

    keep_centroids = []
    keep_scores    = []
    suppressed     = np.zeros(len(dets), dtype=bool)

    for i in order:
        if suppressed[i]:
            continue
        zi, yi, xi = dets[i]
        cluster_idx = []
        for j in range(len(dets)):
            if suppressed[j]:
                continue
            zj, yj, xj = dets[j]
            if abs(zi - zj) <= radius_z and \
               np.sqrt((yi - yj)**2 + (xi - xj)**2) <= radius_xy:
                cluster_idx.append(j)
                suppressed[j] = True

        c_dets = dets[cluster_idx]
        c_scrs = scrs[cluster_idx]
        tw     = c_scrs.sum()

        if tw > 0:
            z_wt = float(np.dot(c_scrs, c_dets[:, 0]) / tw)
            y_wt = float(np.dot(c_scrs, c_dets[:, 1]) / tw)
            x_wt = float(np.dot(c_scrs, c_dets[:, 2]) / tw)
        else:
            z_wt, y_wt, x_wt = float(zi), float(yi), float(xi)

        keep_centroids.append((z_wt, y_wt, x_wt))
        keep_scores.append(float(c_scrs.max()))

    return keep_centroids, keep_scores

print('Detection functions ready.')
print('Using 2.5D stacking (3-channel input) matching training architecture.')

INFER_THRESH=0.25 | MIN_PEAK_DISTANCE=5
MIN_CONSECUTIVE_Z=3
NMS_RADIUS_XY=6.0 | NMS_RADIUS_Z=3
Z_CONSISTENCY_XY=3.0
Detection functions ready.
Using 2.5D stacking (3-channel input) matching training architecture.


In [6]:
def link_timepoints_advanced(nodes_t0, nodes_t1,
                              max_dist_um=MAX_LINK_DIST_UM,
                              div_max_dist_um=DIV_MAX_DIST_UM):
    """
    Advanced linker with:
    - Kinematic prediction (velocity from previous frame)
    - Confidence-weighted cost matrix
    - Division detection with minimum daughter separation check

    MAX_LINK_DIST_UM=15.0 is the linking radius.
    Do NOT use 7.0 here -- 7um is the competition's NODE MATCHING radius
    (how close a predicted node must be to a GT node to count as a match),
    not how far a cell can move between frames.
    """
    if not nodes_t0 or not nodes_t1:
        return []

    pos0_um = np.array([[n['phys_z'], n['phys_y'], n['phys_x']]
                         for n in nodes_t0])
    pos1_um = np.array([[n['phys_z'], n['phys_y'], n['phys_x']]
                         for n in nodes_t1])

    # kinematic prediction: extrapolate from last known velocity
    predicted_pos1_um = []
    for i, n in enumerate(nodes_t0):
        if n.get('parent_pos_um') is not None:
            velocity = np.clip(pos0_um[i] - n['parent_pos_um'], -6.0, 6.0)
            predicted_pos1_um.append(pos0_um[i] + velocity)
        else:
            predicted_pos1_um.append(pos0_um[i])
    predicted_pos1_um = np.array(predicted_pos1_um)

    # confidence-weighted cost matrix
    dist_matrix = np.linalg.norm(
        predicted_pos1_um[:, None, :] - pos1_um[None, :, :], axis=2
    )
    scores_t1          = np.array([n['score'] for n in nodes_t1])
    confidence_penalty = 2.0 * (1.0 - scores_t1)[None, :]
    cost_matrix        = dist_matrix + confidence_penalty

    # hard cutoff using real linking distance (NOT the 7um metric radius)
    cost_matrix[dist_matrix > max_dist_um] = 1e5

    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    links        = []
    matched_curr = set()
    prev_to_curr = {}

    for r, c in zip(row_ind, col_ind):
        if dist_matrix[r, c] <= max_dist_um:
            links.append({
                'source_id':     nodes_t0[r]['id'],
                'target_id':     nodes_t1[c]['id'],
                'source_pos_um': pos0_um[r]
            })
            matched_curr.add(c)
            prev_to_curr.setdefault(r, []).append(c)

    # division check: unmatched t1 nodes close to an already-matched t0 node
    for ci, n1 in enumerate(nodes_t1):
        if ci in matched_curr:
            continue
        best_r, best_d = None, div_max_dist_um
        for ri in range(len(nodes_t0)):
            if ri not in prev_to_curr or len(prev_to_curr[ri]) != 1:
                continue
            d = np.linalg.norm(pos0_um[ri] - pos1_um[ci])
            if d < best_d:
                existing_ci   = prev_to_curr[ri][0]
                daughter_dist = np.linalg.norm(pos1_um[ci] - pos1_um[existing_ci])
                if daughter_dist >= 3.0:   # daughters must be physically separated
                    best_d = d
                    best_r = ri
        if best_r is not None:
            links.append({
                'source_id':     nodes_t0[best_r]['id'],
                'target_id':     nodes_t1[ci]['id'],
                'source_pos_um': pos0_um[best_r]
            })

    return links

print(f'Cell 5 complete. Linker defined.')
print(f'  MAX_LINK_DIST_UM={MAX_LINK_DIST_UM} (cell movement budget per frame)')
print(f'  DIV_MAX_DIST_UM={DIV_MAX_DIST_UM} (division parent-daughter search radius)')

Cell 5 complete. Linker defined.
  MAX_LINK_DIST_UM=15.0 (cell movement budget per frame)
  DIV_MAX_DIST_UM=8.0 (division parent-daughter search radius)


In [7]:
import networkx as nx

test_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')
)
print(f'Test datasets: {len(test_names)}')
for n in test_names:
    print(f'  {n}')

all_rows = []

for folder_name in test_names:
    zp   = str(TEST_DIR / f'{folder_name}.zarr')
    meta = read_meta(zp)
    n_t  = meta['shape'][0]
    print(f'\n{folder_name} | {n_t} timepoints | shape {meta["shape"]}')

    all_nodes_by_t     = {}
    edges_list         = []
    global_node_counter = 0

    for t in range(n_t):
        vol  = load_vol(zp, t, meta)
        pvol = pool_xy(vol)

        confirmed, raw_scores = detect_volume_with_scores(pvol, model)

        # FIX: unpack BOTH return values from nms_3d_weighted
        # nms_scores[i] is aligned with nuclei[i] -- do NOT use raw_scores[i]
        nuclei, nms_scores = nms_3d_weighted(confirmed, raw_scores)

        nodes_t = []
        for i, (z, py, px) in enumerate(nuclei):
            orig_y = py * POOL
            orig_x = px * POOL
            phys_z = z      * SCALE[0]
            phys_y = orig_y * SCALE[1]
            phys_x = orig_x * SCALE[2]

            nodes_t.append({
                'id':            global_node_counter,
                't':             t,
                'z':             z,
                'y':             orig_y,
                'x':             orig_x,
                'phys_z':        phys_z,
                'phys_y':        phys_y,
                'phys_x':        phys_x,
                'score':         nms_scores[i],   # FIX: aligned score, not raw_scores[i]
                'parent_pos_um': None,
            })
            global_node_counter += 1

        all_nodes_by_t[t] = nodes_t

        if t % 20 == 0 or t == n_t - 1:
            print(f'  t={t:3d}/{n_t-1} | {len(nuclei)} nuclei detected')

        # link to previous frame
        if t > 0:
            links = link_timepoints_advanced(
                all_nodes_by_t[t - 1], all_nodes_by_t[t]
            )
            for link in links:
                edges_list.append((link['source_id'], link['target_id']))
                # propagate position for next frame's velocity prediction
                for n in all_nodes_by_t[t]:
                    if n['id'] == link['target_id']:
                        n['parent_pos_um'] = link['source_pos_um']
                        break

    # filter short tracks using NetworkX
    G = nx.Graph()
    for t, nodes in all_nodes_by_t.items():
        for n in nodes:
            G.add_node(n['id'], data=n)
    for src, tgt in edges_list:
        G.add_edge(src, tgt)

    keep_set = set()
    for comp in nx.connected_components(G):
        if len(comp) >= 3:
            keep_set.update(comp)

    # node rows
    for t, nodes in all_nodes_by_t.items():
        for n in nodes:
            if n['id'] in keep_set:
                all_rows.append({
                    'dataset':   folder_name,
                    'row_type':  'node',
                    'node_id':   n['id'],
                    't':         n['t'],
                    'z':         int(round(n['z'])),
                    'y':         int(round(n['y'])),
                    'x':         int(round(n['x'])),
                    'source_id': -1,
                    'target_id': -1,
                })

    # edge rows
    for src, tgt in edges_list:
        if src in keep_set and tgt in keep_set:
            all_rows.append({
                'dataset':   folder_name,
                'row_type':  'edge',
                'node_id':   -1,
                't':         -1,
                'z':         -1,
                'y':         -1,
                'x':         -1,
                'source_id': src,
                'target_id': tgt,
            })

    n_nodes = sum(1 for r in all_rows
                  if r['dataset'] == folder_name and r['row_type'] == 'node')
    n_edges = sum(1 for r in all_rows
                  if r['dataset'] == folder_name and r['row_type'] == 'edge')
    print(f'  -> kept: {n_nodes} nodes | {n_edges} edges')

# write submission
submission = pd.DataFrame(
    all_rows,
    columns=['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
             'source_id', 'target_id']
)
submission.index.name = 'id'
sub_path = OUT_DIR / 'submission.csv'
submission.to_csv(sub_path)

print(f'\n{"="*50}')
print(f'submission.csv written: {sub_path}')
print(f'  Total rows : {len(submission)}')
print(f'  Node rows  : {(submission.row_type == "node").sum()}')
print(f'  Edge rows  : {(submission.row_type == "edge").sum()}')
print(f'  Datasets   : {submission.dataset.nunique()}')
print(f'{"="*50}')

assert set(submission.columns) == {
    'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
    'source_id', 'target_id'
}
assert submission.index.name == 'id'
assert submission[submission.row_type == 'node']['source_id'].eq(-1).all()
assert submission[submission.row_type == 'node']['target_id'].eq(-1).all()
assert submission[submission.row_type == 'edge']['node_id'].eq(-1).all()
print('Format sanity checks passed.')

Test datasets: 4
  44b6_0113de3b
  44b6_0b24845f
  6bba_05b6850b
  6bba_05db0fb1

44b6_0113de3b | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 280 nuclei detected
  t= 20/99 | 285 nuclei detected
  t= 40/99 | 288 nuclei detected
  t= 60/99 | 294 nuclei detected
  t= 80/99 | 299 nuclei detected
  t= 99/99 | 298 nuclei detected
  -> kept: 28051 nodes | 27123 edges

44b6_0b24845f | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 347 nuclei detected
  t= 20/99 | 299 nuclei detected
  t= 40/99 | 301 nuclei detected
  t= 60/99 | 373 nuclei detected
  t= 80/99 | 402 nuclei detected
  t= 99/99 | 374 nuclei detected
  -> kept: 33920 nodes | 33096 edges

6bba_05b6850b | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 221 nuclei detected
  t= 20/99 | 194 nuclei detected
  t= 40/99 | 205 nuclei detected
  t= 60/99 | 193 nuclei detected
  t= 80/99 | 200 nuclei detected
  t= 99/99 | 201 nuclei detected
  -> kept: 19824 nodes | 18979 edges

6bba_05db0fb1 | 100 timepoints | 